# Session Report Utilities

Shared, fixed analytical definitions for session-level and future comparative reports.

In [ ]:
import os
import sqlite3

import numpy as np
import pandas as pd

# Keep these boundaries stable so altitude results remain comparable between sessions.
ALTITUDE_BAND_LABELS = ['Below 5,000 ft', '5,000–14,999 ft', '15,000–29,999 ft', '30,000 ft and above']
ALTITUDE_BAND_EDGES = [-np.inf, 5000, 15000, 30000, np.inf]

# Use eight equal 45-degree sectors centred on the cardinal and intercardinal directions.
HEADING_LABELS = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
HEADING_EDGES = [-22.5, 22.5, 67.5, 112.5, 157.5, 202.5, 247.5, 292.5, 337.5]

# Rates within 100 feet per minute of zero are treated as level for descriptive reporting.
LEVEL_RATE_THRESHOLD_FPM = 100

In [ ]:
def query_optional_data(site, query):
    """
    Execute a read-only report query without rejecting a valid empty result.

    :param site: Database-site key from the shared DB_PATH_VARIABLES mapping.
    :param query: Complete read-only SQL query to execute.
    :return: A pandas DataFrame, which may contain zero rows.
    """
    # Open a short-lived SQLite connection and preserve result columns even when no rows match.
    database_path = os.environ[DB_PATH_VARIABLES[site]]
    with sqlite3.connect(database_path) as connection:
        return pd.read_sql_query(query, connection)

In [ ]:
def assign_altitude_band(values, include_unknown=True):
    """
    Classify altitude values using the shared fixed bands.

    :param values: Series-like altitude values measured in feet.
    :param include_unknown: Whether missing values should be labelled Unknown.
    :return: A pandas Series containing the ordered altitude-band labels.
    """
    # Convert invalid inputs to missing values before applying fixed boundaries.
    numeric = pd.to_numeric(values, errors='coerce')
    bands = pd.cut(numeric, bins=ALTITUDE_BAND_EDGES, labels=ALTITUDE_BAND_LABELS, right=False)
    return bands.astype('object').fillna('Unknown') if include_unknown else bands


def assign_heading_sector(values):
    """
    Classify headings into eight fixed compass sectors.

    :param values: Series-like heading values measured clockwise in degrees from north.
    :return: A pandas Series containing compass labels or Unknown.
    """
    # Wrap valid headings into 0–360 degrees, then shift north-sector values for binning.
    headings = pd.to_numeric(values, errors='coerce') % 360
    shifted = (headings + 22.5) % 360
    sectors = pd.cut(shifted, bins=np.arange(0, 361, 45), labels=HEADING_LABELS, right=False)
    return sectors.astype('object').fillna('Unknown')


def assign_vertical_behaviour(values):
    """
    Classify vertical rates as climbing, level, descending, or unknown.

    :param values: Series-like vertical rates measured in feet per minute.
    :return: A pandas Series containing vertical-behaviour labels.
    """
    # Apply a symmetric dead band around zero to avoid labelling small fluctuations as climbs or descents.
    rates = pd.to_numeric(values, errors='coerce')
    result = pd.Series('Unknown', index=rates.index, dtype='object')
    result.loc[rates > LEVEL_RATE_THRESHOLD_FPM] = 'Climbing'
    result.loc[rates < -LEVEL_RATE_THRESHOLD_FPM] = 'Descending'
    result.loc[rates.between(-LEVEL_RATE_THRESHOLD_FPM, LEVEL_RATE_THRESHOLD_FPM)] = 'Level'
    return result